# DQN for 2048 — simple, strong, and easy to optimize

Bản này giữ đúng tinh thần mạnh của notebook baseline đơn giản:

- **raw observation** từ OpenSpiel, không thêm CNN / one-hot / reward shaping
- **MLP 2 hidden layers (256-256)**, dễ train
- **legal-action masking**
- **train from scratch**
- chỉ thêm các ổn định hóa nhẹ, ít rủi ro:
  - **Double DQN**
  - **Huber loss**
  - **fixed validation/test seeds** để chọn model công bằng hơn
  - **checkpoint + resume**


In [ ]:
# Colab / notebook setup
!python -V
!pip -q install --upgrade pip
!pip -q install open-spiel torch matplotlib imageio tqdm

In [ ]:
import os
import json
import random
import re
from collections import deque, namedtuple

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pyspiel

print("PyTorch version:", torch.__version__)
print("OpenSpiel version:", pyspiel.__version__ if hasattr(pyspiel, "__version__") else "unknown")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("CUDA available:", torch.cuda.is_available())
print("MPS available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
print("Device:", DEVICE)

## Inspect OpenSpiel 2048

In [ ]:
game = pyspiel.load_game("2048")
state = game.new_initial_state()

game_type = game.get_type()
short_name = game_type.short_name if hasattr(game_type, "short_name") else "2048"

print("Registered game:", short_name)
print("Num distinct actions:", game.num_distinct_actions())
print("Observation tensor shape:", game.observation_tensor_shape())
print("Observation tensor size:", game.observation_tensor_size())
print("Max chance outcomes:", game.max_chance_outcomes())
print("Max game length:", game.max_game_length())
print("Min / Max utility:", game.min_utility(), game.max_utility())
print()
print("Initial state is chance node:", state.is_chance_node())
print("Initial state string:")
print(state)

## Helpers for observation, chance nodes, and board parsing

In [ ]:
def extract_obs(state, player_id=0):
    """Return a flat float32 observation vector for the player."""
    for fn_name, args in [
        ("observation_tensor", (player_id,)),
        ("observation_tensor", tuple()),
        ("information_state_tensor", (player_id,)),
        ("information_state_tensor", tuple()),
    ]:
        fn = getattr(state, fn_name, None)
        if fn is None:
            continue
        try:
            obs = fn(*args)
            obs = np.asarray(obs, dtype=np.float32).reshape(-1)
            return obs
        except TypeError:
            pass
    raise RuntimeError("Could not extract an observation tensor from state.")


def legal_actions(state, player_id=0):
    try:
        return list(state.legal_actions(player_id))
    except TypeError:
        return list(state.legal_actions())


def sample_chance_action(state, rng):
    outcomes = state.chance_outcomes()
    actions, probs = zip(*outcomes)
    idx = rng.choice(len(actions), p=np.asarray(probs, dtype=np.float64))
    return actions[idx]


def auto_resolve_chance_nodes(state, rng):
    while state.is_chance_node() and not state.is_terminal():
        a = sample_chance_action(state, rng)
        state.apply_action(a)
    return state


def state_return(state, player_id=0):
    vals = state.returns()
    return float(vals[player_id]) if len(vals) > player_id else 0.0


def state_reward(state, player_id=0):
    vals = state.rewards()
    return float(vals[player_id]) if len(vals) > player_id else 0.0


def parse_board_numbers(state):
    """Best-effort parser for showing the board as a 4x4 integer array."""
    txt = str(state)
    nums = [int(x) for x in re.findall(r"\d+", txt)]
    if len(nums) >= 16:
        nums = nums[-16:]
        return np.array(nums, dtype=np.int64).reshape(4, 4)
    return None


test_state = game.new_initial_state()
auto_resolve_chance_nodes(test_state, np.random.default_rng(0))
print("Observation shape after resolving initial chance:", extract_obs(test_state).shape)
print("Legal actions:", legal_actions(test_state))
print("Board (best effort):")
print(parse_board_numbers(test_state))
print()
print(test_state)

## Lightweight environment wrapper

In [ ]:
class OpenSpiel2048Env:
    def __init__(self, seed=42):
        self.game = pyspiel.load_game("2048")
        self.player_id = 0
        self.num_actions = self.game.num_distinct_actions()
        self.obs_dim = self.game.observation_tensor_size()
        self.rng = np.random.default_rng(seed)
        self.state = None

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.state = self.game.new_initial_state()
        auto_resolve_chance_nodes(self.state, self.rng)
        return extract_obs(self.state, self.player_id)

    def step(self, action):
        if self.state is None:
            raise RuntimeError("Call reset() before step().")
        if self.state.is_terminal():
            raise RuntimeError("Episode already ended. Call reset().")

        legal = legal_actions(self.state, self.player_id)
        if action not in legal:
            raise ValueError(f"Illegal action {action}. Legal actions: {legal}")

        prev_return = state_return(self.state, self.player_id)

        self.state.apply_action(int(action))
        auto_resolve_chance_nodes(self.state, self.rng)

        done = self.state.is_terminal()
        next_obs = (
            extract_obs(self.state, self.player_id)
            if not done else np.zeros(self.obs_dim, dtype=np.float32)
        )
        new_return = state_return(self.state, self.player_id)

        reward = new_return - prev_return
        info = {
            "legal_actions": legal_actions(self.state, self.player_id) if not done else [],
            "state_return": new_return,
            "state_reward_raw": state_reward(self.state, self.player_id),
            "board": parse_board_numbers(self.state),
            "state_text": str(self.state),
        }
        return next_obs, float(reward), done, info

    def legal_actions(self):
        if self.state is None or self.state.is_terminal():
            return []
        return legal_actions(self.state, self.player_id)

    def render(self):
        if self.state is None:
            print("<env not reset>")
        else:
            print(self.state)

## Model and replay buffer

Giữ backbone thật đơn giản: **MLP 256-256** trên raw observation.  
Không dùng CNN, one-hot board, reward shaping, hoặc feature thủ công.


In [ ]:
Transition = namedtuple(
    "Transition",
    ["obs", "action", "reward", "next_obs", "done", "legal_mask", "next_legal_mask"],
)

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def __len__(self):
        return len(self.buffer)

    def add(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))


def make_legal_mask(num_actions, legal_actions_list):
    mask = np.zeros(num_actions, dtype=np.float32)
    mask[legal_actions_list] = 1.0
    return mask


class QNetwork(nn.Module):
    def __init__(self, obs_dim, num_actions, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions),
        )

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def masked_greedy_action(q_net, obs, legal_actions_list, num_actions, epsilon=0.0, device=DEVICE):
    if random.random() < epsilon:
        return random.choice(legal_actions_list)

    obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    q = q_net(obs_t).squeeze(0)

    legal_mask = torch.zeros(num_actions, dtype=torch.bool, device=device)
    legal_mask[legal_actions_list] = True

    q_masked = q.masked_fill(~legal_mask, -1e9)
    action = int(torch.argmax(q_masked).item())
    return action

## Hyperparameters

Tinh thần của bản này là:
- model nhỏ, học sớm
- epsilon decay đủ chậm để train lâu hơn
- không làm pipeline quá nặng


In [ ]:
# Reproducibility
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Training hyperparameters
NUM_EPISODES = 5_000
BUFFER_SIZE = 100_000
BATCH_SIZE = 128
GAMMA = 0.99
LR = 3e-4
TARGET_SYNC_EVERY = 1_000
LEARN_START = 2_000
LEARN_EVERY = 4
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY_STEPS = 100_000
MAX_STEPS_PER_EPISODE = 5_000
GRAD_CLIP = 10.0
HIDDEN_DIM = 256

# Evaluation protocol
EVAL_EVERY = 50
VAL_SEEDS = list(range(10_000, 10_020))   # fixed validation seeds
TEST_SEEDS = list(range(20_000, 20_050))  # fixed test seeds

# Checkpointing / resume
CHECKPOINT_DIR = "checkpoints_2048_simple_spirit"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
LATEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "latest_resume.pt")
BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")
SAVE_EVERY = 250
RESUME = False   # set True to continue training from LATEST_CKPT_PATH

train_env = OpenSpiel2048Env(seed=SEED)
obs_dim = train_env.obs_dim
num_actions = train_env.num_actions

q_net = QNetwork(obs_dim, num_actions, hidden_dim=HIDDEN_DIM).to(DEVICE)
target_net = QNetwork(obs_dim, num_actions, hidden_dim=HIDDEN_DIM).to(DEVICE)
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer = optim.Adam(q_net.parameters(), lr=LR)
replay = ReplayBuffer(BUFFER_SIZE)

print("obs_dim =", obs_dim)
print("num_actions =", num_actions)
print("device =", DEVICE)

## Training helpers

Khác với baseline gốc ở 2 chỗ nhỏ nhưng hữu ích:
- **Double DQN target**
- **Huber loss** thay cho MSE


In [ ]:
def epsilon_by_step(step):
    frac = min(1.0, step / EPS_DECAY_STEPS)
    return EPS_START + frac * (EPS_END - EPS_START)


def dqn_update(batch):
    obs = torch.tensor(np.asarray(batch.obs), dtype=torch.float32, device=DEVICE)
    actions = torch.tensor(batch.action, dtype=torch.int64, device=DEVICE).unsqueeze(1)
    rewards = torch.tensor(batch.reward, dtype=torch.float32, device=DEVICE)
    next_obs = torch.tensor(np.asarray(batch.next_obs), dtype=torch.float32, device=DEVICE)
    dones = torch.tensor(batch.done, dtype=torch.float32, device=DEVICE)

    next_legal_mask = torch.tensor(np.asarray(batch.next_legal_mask), dtype=torch.bool, device=DEVICE)

    q_values = q_net(obs)
    q_sa = q_values.gather(1, actions).squeeze(1)

    with torch.no_grad():
        # Double DQN:
        # 1) online net selects argmax action under legal mask
        next_q_online = q_net(next_obs)
        next_q_online = next_q_online.masked_fill(~next_legal_mask, -1e9)
        next_actions = torch.argmax(next_q_online, dim=1, keepdim=True)

        # 2) target net evaluates that action
        next_q_target = target_net(next_obs).gather(1, next_actions).squeeze(1)
        next_q_target = torch.where(dones > 0.5, torch.zeros_like(next_q_target), next_q_target)

        target = rewards + GAMMA * next_q_target

    loss = F.smooth_l1_loss(q_sa, target)

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(q_net.parameters(), GRAD_CLIP)
    optimizer.step()

    return float(loss.item())


@torch.no_grad()
def evaluate_policy(model, seeds, max_steps=MAX_STEPS_PER_EPISODE):
    returns = []
    lengths = []
    max_tiles = []

    for seed in seeds:
        env = OpenSpiel2048Env(seed=seed)
        obs = env.reset(seed=seed)
        done = False
        ep_return = 0.0
        ep_len = 0
        last_board = None

        while not done and ep_len < max_steps:
            legal = env.legal_actions()
            action = masked_greedy_action(model, obs, legal, env.num_actions, epsilon=0.0, device=DEVICE)
            obs, reward, done, info = env.step(action)
            ep_return += reward
            ep_len += 1
            if info.get("board") is not None:
                last_board = info["board"]

        returns.append(float(ep_return))
        lengths.append(int(ep_len))
        max_tiles.append(int(last_board.max()) if last_board is not None else 0)

    returns = np.asarray(returns, dtype=np.float32)
    lengths = np.asarray(lengths, dtype=np.float32)
    max_tiles = np.asarray(max_tiles, dtype=np.int64)

    return {
        "mean_return": float(np.mean(returns)),
        "std_return": float(np.std(returns)),
        "mean_length": float(np.mean(lengths)),
        "mean_max_tile": float(np.mean(max_tiles)),
        "returns": returns,
        "lengths": lengths,
        "max_tiles": max_tiles,
    }


def summarize_tiles(max_tiles):
    levels = [64, 128, 256, 512, 1024, 2048]
    summary = {}
    n = len(max_tiles)
    for lvl in levels:
        summary[lvl] = float(np.mean(max_tiles >= lvl)) if n > 0 else 0.0
    return summary

## Checkpoint / resume helpers

In [ ]:
def save_resume_checkpoint(path, episode, global_step, best_eval_mean, best_state_dict,
                           episode_returns, episode_lengths, loss_history, val_history):
    ckpt = {
        "episode": episode,
        "global_step": global_step,
        "model_state_dict": q_net.state_dict(),
        "target_state_dict": target_net.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "replay_buffer": list(replay.buffer),
        "best_eval_mean": best_eval_mean,
        "best_state_dict": best_state_dict,
        "episode_returns": episode_returns,
        "episode_lengths": episode_lengths,
        "loss_history": loss_history,
        "val_history": val_history,
        "config": {
            "SEED": SEED,
            "NUM_EPISODES": NUM_EPISODES,
            "BUFFER_SIZE": BUFFER_SIZE,
            "BATCH_SIZE": BATCH_SIZE,
            "GAMMA": GAMMA,
            "LR": LR,
            "TARGET_SYNC_EVERY": TARGET_SYNC_EVERY,
            "LEARN_START": LEARN_START,
            "LEARN_EVERY": LEARN_EVERY,
            "EPS_START": EPS_START,
            "EPS_END": EPS_END,
            "EPS_DECAY_STEPS": EPS_DECAY_STEPS,
            "MAX_STEPS_PER_EPISODE": MAX_STEPS_PER_EPISODE,
            "GRAD_CLIP": GRAD_CLIP,
            "HIDDEN_DIM": HIDDEN_DIM,
        },
        "rng_state": {
            "python_random": random.getstate(),
            "numpy_random": np.random.get_state(),
            "torch_random": torch.get_rng_state(),
            "torch_cuda_random": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
    }
    torch.save(ckpt, path)


def load_resume_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE)

    q_net.load_state_dict(ckpt["model_state_dict"])
    target_net.load_state_dict(ckpt["target_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)

    replay.buffer.clear()
    replay.buffer.extend(ckpt["replay_buffer"])

    if "rng_state" in ckpt:
        random.setstate(ckpt["rng_state"]["python_random"])
        np.random.set_state(ckpt["rng_state"]["numpy_random"])
        torch.set_rng_state(ckpt["rng_state"]["torch_random"])
        if torch.cuda.is_available() and ckpt["rng_state"]["torch_cuda_random"] is not None:
            torch.cuda.set_rng_state_all(ckpt["rng_state"]["torch_cuda_random"])

    return ckpt

## Train

Điểm khác với notebook bạn đang có là **best model được chọn trên fixed validation seeds**.  
Như vậy `best` bớt bị phồng do may mắn của seed.


In [ ]:
episode_returns = []
episode_lengths = []
loss_history = []
val_history = []

global_step = 0
best_eval_mean = -float("inf")
best_state_dict = None
start_episode = 1

if RESUME and os.path.exists(LATEST_CKPT_PATH):
    ckpt = load_resume_checkpoint(LATEST_CKPT_PATH)
    episode_returns = ckpt["episode_returns"]
    episode_lengths = ckpt["episode_lengths"]
    loss_history = ckpt["loss_history"]
    val_history = ckpt["val_history"]
    global_step = ckpt["global_step"]
    best_eval_mean = ckpt["best_eval_mean"]
    best_state_dict = ckpt["best_state_dict"]
    start_episode = ckpt["episode"] + 1
    print(f"Resumed from {LATEST_CKPT_PATH} | start_episode={start_episode} | global_step={global_step} | replay={len(replay)}")
else:
    print("Starting fresh training.")

for episode in tqdm(range(start_episode, NUM_EPISODES + 1), desc="Training"):
    obs = train_env.reset(seed=SEED + episode)
    done = False
    ep_return = 0.0
    ep_len = 0

    while not done and ep_len < MAX_STEPS_PER_EPISODE:
        eps = epsilon_by_step(global_step)
        legal = train_env.legal_actions()
        legal_mask = make_legal_mask(num_actions, legal)

        action = masked_greedy_action(
            q_net=q_net,
            obs=obs,
            legal_actions_list=legal,
            num_actions=num_actions,
            epsilon=eps,
            device=DEVICE,
        )

        next_obs, reward, done, info = train_env.step(action)
        next_legal = info["legal_actions"] if not done else []
        next_legal_mask = make_legal_mask(num_actions, next_legal)

        replay.add(obs, action, reward, next_obs, done, legal_mask, next_legal_mask)

        obs = next_obs
        ep_return += reward
        ep_len += 1
        global_step += 1

        if len(replay) >= max(LEARN_START, BATCH_SIZE) and global_step % LEARN_EVERY == 0:
            batch = replay.sample(BATCH_SIZE)
            loss = dqn_update(batch)
            loss_history.append(loss)

        if global_step % TARGET_SYNC_EVERY == 0:
            target_net.load_state_dict(q_net.state_dict())

    episode_returns.append(ep_return)
    episode_lengths.append(ep_len)

    if episode % EVAL_EVERY == 0:
        metrics = evaluate_policy(q_net, VAL_SEEDS)
        tile_summary = summarize_tiles(metrics["max_tiles"])
        val_history.append({
            "episode": episode,
            "mean_return": metrics["mean_return"],
            "std_return": metrics["std_return"],
            "mean_length": metrics["mean_length"],
            "mean_max_tile": metrics["mean_max_tile"],
            "tile_summary": tile_summary,
        })

        print(
            f"[Val] ep={episode:6d} | mean={metrics['mean_return']:.1f} | "
            f"std={metrics['std_return']:.1f} | len={metrics['mean_length']:.1f} | "
            f"mean_max_tile={metrics['mean_max_tile']:.1f} | "
            f">=256={100*tile_summary[256]:.1f}% | >=512={100*tile_summary[512]:.1f}%"
        )

        if metrics["mean_return"] > best_eval_mean:
            best_eval_mean = metrics["mean_return"]
            best_state_dict = {k: v.detach().cpu().clone() for k, v in q_net.state_dict().items()}
            torch.save(
                {
                    "model_state_dict": best_state_dict,
                    "episode": episode,
                    "best_val_mean_return": best_eval_mean,
                    "val_metrics": metrics,
                },
                BEST_CKPT_PATH,
            )
            print(f"  -> new best model saved to {BEST_CKPT_PATH}")

    if episode % SAVE_EVERY == 0:
        save_resume_checkpoint(
            LATEST_CKPT_PATH, episode, global_step, best_eval_mean, best_state_dict,
            episode_returns, episode_lengths, loss_history, val_history
        )

save_resume_checkpoint(
    LATEST_CKPT_PATH, episode, global_step, best_eval_mean, best_state_dict,
    episode_returns, episode_lengths, loss_history, val_history
)

print("Training complete.")
print("Best validation mean return:", best_eval_mean)

## Training curves

In [ ]:
def moving_average(x, w=50):
    x = np.asarray(x, dtype=np.float32)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(episode_returns, alpha=0.30, label="episode return")
ma = moving_average(episode_returns, 50)
plt.plot(range(len(ma)), ma, label="moving avg (50)")
plt.title("Training return")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(episode_lengths, alpha=0.5)
plt.title("Episode length")
plt.xlabel("Episode")
plt.ylabel("Steps")

plt.subplot(1, 3, 3)
plt.plot(loss_history, alpha=0.8)
plt.title("DQN loss (Huber)")
plt.xlabel("Update step")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()

if val_history:
    val_eps = [x["episode"] for x in val_history]
    val_means = [x["mean_return"] for x in val_history]
    val_stds = [x["std_return"] for x in val_history]

    plt.figure(figsize=(7, 4))
    plt.plot(val_eps, val_means, label="validation mean return")
    plt.fill_between(
        val_eps,
        np.asarray(val_means) - np.asarray(val_stds),
        np.asarray(val_means) + np.asarray(val_stds),
        alpha=0.2,
        label="±1 std",
    )
    plt.title("Fixed-seed validation performance")
    plt.xlabel("Episode")
    plt.ylabel("Return")
    plt.legend()
    plt.show()

## Load the best validation checkpoint and evaluate on fixed test seeds

In [ ]:
if os.path.exists(BEST_CKPT_PATH):
    best_ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
    q_net.load_state_dict(best_ckpt["model_state_dict"])
    target_net.load_state_dict(best_ckpt["model_state_dict"])
    print("Loaded best validation checkpoint from:", BEST_CKPT_PATH)
    print("Best validation mean return:", best_ckpt["best_val_mean_return"])
else:
    print("Best checkpoint not found; using current q_net weights.")

test_metrics = evaluate_policy(q_net, TEST_SEEDS)
test_tile_summary = summarize_tiles(test_metrics["max_tiles"])

print(f"Test mean return: {test_metrics['mean_return']:.1f}")
print(f"Test std return: {test_metrics['std_return']:.1f}")
print(f"Test mean length: {test_metrics['mean_length']:.1f}")
print(f"Test mean max tile: {test_metrics['mean_max_tile']:.1f}")
for lvl in [64, 128, 256, 512, 1024, 2048]:
    print(f"P(max_tile >= {lvl:4d}): {100*test_tile_summary[lvl]:5.1f}%")

## One greedy rollout (for qualitative inspection only)

Rollout đơn lẻ chỉ để nhìn board cuối; **không** dùng làm metric quyết định.


In [ ]:
eval_seed = TEST_SEEDS[0]
eval_env = OpenSpiel2048Env(seed=eval_seed)
obs = eval_env.reset(seed=eval_seed)
done = False
greedy_return = 0.0
rollout = []

while not done and len(rollout) < MAX_STEPS_PER_EPISODE:
    legal = eval_env.legal_actions()
    action = masked_greedy_action(q_net, obs, legal, num_actions, epsilon=0.0, device=DEVICE)
    next_obs, reward, done, info = eval_env.step(action)
    rollout.append({
        "action": action,
        "reward": reward,
        "legal_actions": legal,
        "board": info["board"],
        "state_text": info["state_text"],
    })
    obs = next_obs
    greedy_return += reward

print("Greedy evaluation return:", greedy_return)
print("Rollout length:", len(rollout))
if rollout and rollout[-1]["board"] is not None:
    print("Max tile reached:", int(np.max(rollout[-1]["board"])))
    print()
    print(rollout[-1]["board"])
print()
eval_env.render()

In [ ]:
# Show a few last boards from the greedy rollout
n_show = min(5, len(rollout))
for i, step_info in enumerate(rollout[-n_show:], start=len(rollout)-n_show+1):
    print("=" * 60)
    print(f"Step {i} | action={step_info['action']} | reward={step_info['reward']:.1f}")
    if step_info["board"] is not None:
        print(step_info["board"])
    print(step_info["state_text"])

## Save a compact final artifact

In [ ]:
final_artifact_path = "dqn_openspiel_2048_simple_spirit.pt"
torch.save(
    {
        "model_state_dict": q_net.state_dict(),
        "target_state_dict": target_net.state_dict(),
        "obs_dim": obs_dim,
        "num_actions": num_actions,
        "hidden_dim": HIDDEN_DIM,
        "device": str(DEVICE),
        "episode_returns": episode_returns,
        "episode_lengths": episode_lengths,
        "loss_history": loss_history,
        "val_history": val_history,
        "test_metrics": test_metrics,
        "test_tile_summary": test_tile_summary,
    },
    final_artifact_path,
)
print("Saved compact model artifact to:", final_artifact_path)